In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

pd.set_option("display.float_format", lambda x: f"{x:.4f}")

In [2]:
df = pd.read_csv("/content/aula04_painel.csv")

In [3]:
print("Dimensões:", df.shape)
print("\nColunas:")
print(df.columns.tolist())

print("\nAnos:")
print(df["ano"].value_counts().sort_index())

print("\nGrupos:")
print(df["grupo"].value_counts())

print("\nCruzamento grupo × ano:")
display(pd.crosstab(df["grupo"], df["ano"]))

print("\nValores ausentes nos desfechos:")
display(
    df[["attend_med", "dropout_med", "grade_now_med"]]
    .isna()
    .sum()
    .to_frame("n_missing")
)

Dimensões: (19425, 19)

Colunas:
['n_criancas', 'attend_med', 'dropout_med', 'grade_now_med', 'ano', 'cod_dtm', 'grupo_ifpri', 'regiao', 'uf', 'n_criancas_05', 'renda_pc_05', 'educ_chefe_m_05', 'educ_chefe_f_05', 'idade_chefe_m_05', 'idade_chefe_f_05', 'sitdom_05', 'grupo', 'pos', 'tratado']

Anos:
ano
2005    11499
2009     7926
Name: count, dtype: int64

Grupos:
grupo
C2    9392
T1    5879
C1    4154
Name: count, dtype: int64

Cruzamento grupo × ano:


ano,2005,2009
grupo,,
C1,2382,1772
C2,5731,3661
T1,3386,2493



Valores ausentes nos desfechos:


,n_missing
attend_med,475
dropout_med,1157
grade_now_med,2963


In [4]:
display(
    df[["ano", "grupo", "pos", "tratado"]]
    .drop_duplicates()
    .sort_values(["grupo", "ano"])
)

,ano,grupo,pos,tratado
10,2005,C1,0,0
11510,2009,C1,1,0
3,2005,C2,0,0
11502,2009,C2,1,0
0,2005,T1,0,1
11499,2009,T1,1,1


In [5]:
did_t1_c1 = df[df["grupo"].isin(["T1", "C1"])].copy()

In [6]:
def tabela_did(dados, desfecho):
    tabela = (
        dados
        .groupby(["grupo", "ano"])[desfecho]
        .mean()
        .unstack()
    )

    tabela["Mudança"] = tabela[2009] - tabela[2005]

    did = (
        tabela.loc["T1", "Mudança"]
        - tabela.loc["C1", "Mudança"]
    )

    return tabela, did

In [7]:
tab_attend, did_attend = tabela_did(did_t1_c1, "attend_med")

display(tab_attend)
print(f"DiD attend_med = {did_attend:.4f}")

ano,2005,2009,Mudança
grupo,,,
C1,0.9209,0.8747,-0.0462
T1,0.9339,0.8965,-0.0374


DiD attend_med = 0.0089


In [8]:
desfechos = ["attend_med", "dropout_med", "grade_now_med"]

for y in desfechos:
    tabela, did = tabela_did(did_t1_c1, y)

    print(f"\n===== {y} =====")
    display(tabela)
    print(f"DiD = {did:.4f}")


===== attend_med =====


ano,2005,2009,Mudança
grupo,,,
C1,0.9209,0.8747,-0.0462
T1,0.9339,0.8965,-0.0374


DiD = 0.0089

===== dropout_med =====


ano,2005,2009,Mudança
grupo,,,
C1,0.0388,0.0929,0.0541
T1,0.0312,0.0773,0.0461


DiD = -0.0080

===== grade_now_med =====


ano,2005,2009,Mudança
grupo,,,
C1,4.1168,4.4813,0.3645
T1,4.0296,4.5468,0.5172


DiD = 0.1527


## Item 1 — DiD canônico: T1 vs C1

Nesta etapa, estimamos manualmente o efeito por Diferenças em Diferenças (DiD), comparando o grupo tratado **T1** com o grupo de controle **C1**, nos anos de 2005 e 2009.

O estimador DiD é dado por:

\[
\widehat{\delta}_{DiD}
=
(\bar{Y}_{T1,2009} - \bar{Y}_{T1,2005})
-
(\bar{Y}_{C1,2009} - \bar{Y}_{C1,2005})
\]

A ideia é comparar a mudança temporal observada no grupo tratado com a mudança observada no grupo de controle. Dessa forma, o grupo C1 é utilizado para representar a trajetória contrafactual que T1 teria apresentado na ausência do tratamento.

Os resultados obtidos foram:

| Desfecho | Mudança T1 | Mudança C1 | DiD |
|---|---:|---:|---:|
| `attend_med` | -0,0374 | -0,0462 | 0,0089 |
| `dropout_med` | 0,0461 | 0,0541 | -0,0080 |
| `grade_now_med` | 0,5172 | 0,3645 | 0,1527 |

Para `attend_med`, ambos os grupos apresentaram redução entre 2005 e 2009, mas a queda foi menor em T1. O DiD de 0,0089 corresponde, portanto, a uma evolução relativa favorável de aproximadamente 0,89 ponto percentual para o grupo tratado.

Para `dropout_med`, o abandono aumentou nos dois grupos, porém o aumento foi menor em T1. O DiD de -0,0080 indica uma evolução relativa favorável ao grupo tratado de aproximadamente 0,80 ponto percentual no indicador de abandono.

Por fim, para `grade_now_med`, os dois grupos apresentaram aumento na série/ano escolar médio, mas o crescimento foi maior em T1. O DiD estimado foi de 0,1527.

Nesta etapa, os resultados representam apenas as estimativas pontuais do DiD. A incerteza estatística associada a essas estimativas será analisada por meio da regressão no Item 2.

In [11]:
# Função para estimar a regressão de Diferenças em Diferenças (DiD)
# A especificação utilizada é:
# Y = beta0 + beta1*tratado + beta2*pos + beta3*(tratado*pos) + erro
#
# O coeficiente beta3, associado à interação tratado:pos,
# corresponde ao estimador DiD.
#
# Os erros-padrão são clusterizados por domicílio (cod_dtm),
# pois o mesmo domicílio pode aparecer em mais de um período.

def regressao_did(dados, desfecho):

    modelo = smf.ols(
        formula=f"{desfecho} ~ tratado + pos + tratado:pos",
        data=dados
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": dados.loc[
                dados[desfecho].notna(),
                "cod_dtm"
            ]
        }
    )

    return modelo

In [13]:
# Estima o modelo DiD para cada um dos três desfechos
# e apresenta apenas as estatísticas principais do coeficiente de interação.

for y in desfechos:

    modelo = regressao_did(did_t1_c1, y)

    print(f"\n===== {y} =====")
    print("Coeficiente DiD:", round(modelo.params["tratado:pos"], 4))
    print("Erro-padrão:", round(modelo.bse["tratado:pos"], 4))
    print("p-valor:", round(modelo.pvalues["tratado:pos"], 4))


===== attend_med =====
Coeficiente DiD: 0.0089
Erro-padrão: 0.0107
p-valor: 0.4067

===== dropout_med =====
Coeficiente DiD: -0.008
Erro-padrão: 0.0094
p-valor: 0.3933

===== grade_now_med =====
Coeficiente DiD: 0.1527
Erro-padrão: 0.1038
p-valor: 0.1413


### Item 2 — Confirmação do DiD por regressão

Nesta etapa, estimamos o modelo clássico de Diferenças em Diferenças:

\[
Y_{it} =
\beta_0
+ \beta_1 Tratado_i
+ \beta_2 Pós_t
+ \beta_3 (Tratado_i \times Pós_t)
+ \varepsilon_{it}
\]

O coeficiente de interesse é \(\beta_3\), associado à interação entre tratamento e período pós. Esse coeficiente corresponde ao estimador DiD obtido anteriormente pela tabela 2×2.

Os resultados foram:

- `attend_med`: DiD = 0,0089
- `dropout_med`: DiD = -0,0080
- `grade_now_med`: DiD = 0,1527

Os três coeficientes reproduzem exatamente os valores calculados manualmente no Item 1, confirmando a equivalência entre a formulação algébrica do DiD e sua estimação por regressão.

Os erros-padrão foram clusterizados por domicílio (`cod_dtm`), pois observações do mesmo domicílio em diferentes períodos não devem ser tratadas como independentes.

Nenhum dos três efeitos apresentou significância estatística aos níveis convencionais, uma vez que os p-valores foram superiores a 0,05. Assim, embora os sinais dos coeficientes indiquem uma evolução relativamente favorável de T1 em relação a C1, não há evidência estatística suficiente, nesta especificação, para rejeitar a hipótese nula de efeito igual a zero.

## Item 3 — Teste placebo: C1 vs C2

Nesta etapa, realizamos um teste placebo utilizando apenas os grupos **C1** e **C2**, nenhum dos quais corresponde ao grupo efetivamente tratado na análise principal.

Para reproduzir a estrutura do DiD, definimos C1 como um grupo "pseudo-tratado" e C2 como grupo de controle. Em seguida, estimamos a diferença das diferenças para os três desfechos.

A finalidade do placebo é verificar se o estimador identifica uma diferença temporal relevante mesmo em uma comparação na qual não deveria existir o efeito do tratamento analisado.

Um efeito placebo elevado pode indicar que C1 e C2 seguem trajetórias distintas ao longo do tempo por razões independentes do tratamento, constituindo um sinal de alerta para a comparabilidade entre os grupos.

In [14]:
# ============================================================
# ITEM 3 — PLACEBO: C1 vs C2
# ============================================================
#
# Nesta análise utilizaremos apenas os grupos C1 e C2.
#
# Como nenhum deles corresponde ao grupo tratado T1,
# vamos criar um "pseudo-tratamento":
#
# C1 = 1  -> pseudo-tratado
# C2 = 0  -> controle
#
# A variável 'pos' permanece igual:
# 2005 = 0
# 2009 = 1
#
# O objetivo é verificar se aparece um "efeito" DiD
# mesmo em uma comparação entre dois grupos não tratados.

placebo = df[df["grupo"].isin(["C1", "C2"])].copy()

# Criação da variável de pseudo-tratamento
placebo["pseudo_tratado"] = (placebo["grupo"] == "C1").astype(int)

# Conferência da codificação
display(
    placebo[
        ["ano", "grupo", "pos", "pseudo_tratado"]
    ]
    .drop_duplicates()
    .sort_values(["grupo", "ano"])
)

,ano,grupo,pos,pseudo_tratado
10,2005,C1,0,1
11510,2009,C1,1,1
3,2005,C2,0,0
11502,2009,C2,1,0


In [15]:
# Função geral para calcular o DiD manualmente
#
# Argumentos:
# - dados: base utilizada
# - desfecho: variável de resultado
# - grupo_tratado: grupo considerado tratado na comparação
# - grupo_controle: grupo utilizado como controle
#
# A função calcula:
#
# DiD =
# (mudança do grupo tratado)
# -
# (mudança do grupo controle)

def tabela_did_grupos(
    dados,
    desfecho,
    grupo_tratado,
    grupo_controle
):

    # Calcula a média do desfecho por grupo e ano
    tabela = (
        dados
        .groupby(["grupo", "ano"])[desfecho]
        .mean()
        .unstack()
    )

    # Calcula a mudança entre 2005 e 2009
    tabela["Mudança"] = tabela[2009] - tabela[2005]

    # Calcula a diferença das diferenças
    did = (
        tabela.loc[grupo_tratado, "Mudança"]
        -
        tabela.loc[grupo_controle, "Mudança"]
    )

    return tabela, did

In [16]:
# Calcula o placebo C1 vs C2 manualmente
# para os três desfechos.

for y in desfechos:

    tabela, did = tabela_did_grupos(
        dados=placebo,
        desfecho=y,
        grupo_tratado="C1",
        grupo_controle="C2"
    )

    print(f"\n===== Placebo: {y} =====")
    display(tabela)
    print(f"DiD placebo = {did:.4f}")


===== Placebo: attend_med =====


ano,2005,2009,Mudança
grupo,,,
C1,0.9209,0.8747,-0.0462
C2,0.9304,0.8851,-0.0452


DiD placebo = -0.0010

===== Placebo: dropout_med =====


ano,2005,2009,Mudança
grupo,,,
C1,0.0388,0.0929,0.0541
C2,0.0382,0.0837,0.0455


DiD placebo = 0.0086

===== Placebo: grade_now_med =====


ano,2005,2009,Mudança
grupo,,,
C1,4.1168,4.4813,0.3645
C2,4.8761,5.2187,0.3426


DiD placebo = 0.0219


In [17]:
# Função para estimar a regressão DiD do placebo
#
# Modelo:
#
# Y = beta0
#   + beta1*pseudo_tratado
#   + beta2*pos
#   + beta3*(pseudo_tratado × pos)
#   + erro
#
# O coeficiente beta3 representa o DiD placebo.
#
# Os erros-padrão continuam clusterizados por domicílio.

def regressao_placebo(dados, desfecho):

    modelo = smf.ols(
        formula=(
            f"{desfecho} ~ "
            "pseudo_tratado + pos + pseudo_tratado:pos"
        ),
        data=dados
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": dados.loc[
                dados[desfecho].notna(),
                "cod_dtm"
            ]
        }
    )

    return modelo

In [18]:
# Estima o placebo para cada um dos três desfechos
# e apresenta as estatísticas do coeficiente de interação.

for y in desfechos:

    modelo = regressao_placebo(placebo, y)

    print(f"\n===== Placebo: {y} =====")
    print(
        "Coeficiente DiD:",
        round(modelo.params["pseudo_tratado:pos"], 4)
    )
    print(
        "Erro-padrão:",
        round(modelo.bse["pseudo_tratado:pos"], 4)
    )
    print(
        "p-valor:",
        round(modelo.pvalues["pseudo_tratado:pos"], 4)
    )


===== Placebo: attend_med =====
Coeficiente DiD: -0.001
Erro-padrão: 0.0103
p-valor: 0.9238

===== Placebo: dropout_med =====
Coeficiente DiD: 0.0086
Erro-padrão: 0.009
p-valor: 0.3414

===== Placebo: grade_now_med =====
Coeficiente DiD: 0.0219
Erro-padrão: 0.0992
p-valor: 0.8255


### Interpretação do teste placebo

O teste placebo foi realizado comparando os grupos C1 e C2, tratando C1 como um grupo pseudo-tratado e C2 como controle. Como nenhum desses grupos corresponde ao grupo efetivamente tratado na análise principal, não se espera encontrar um efeito DiD relevante.

Os resultados foram:

| Desfecho | DiD placebo | Erro-padrão | p-valor |
|---|---:|---:|---:|
| `attend_med` | -0,0010 | 0,0103 | 0,9238 |
| `dropout_med` | 0,0086 | 0,0090 | 0,3414 |
| `grade_now_med` | 0,0219 | 0,0992 | 0,8255 |

Em todos os casos, os coeficientes estimados foram pequenos e estatisticamente indistinguíveis de zero aos níveis convencionais.

Assim, o placebo não revelou nenhum resultado especialmente suspeito. Isso sugere que C1 e C2 apresentaram trajetórias temporais relativamente semelhantes para os três desfechos analisados.

É importante notar que o teste placebo não exige que C1 e C2 apresentem os mesmos níveis médios dos desfechos. No caso de `grade_now_med`, por exemplo, os níveis de C2 são substancialmente superiores aos de C1, mas as mudanças entre 2005 e 2009 foram semelhantes. Como o DiD utiliza diferenças temporais, o ponto central é a comparabilidade das trajetórias, e não a igualdade dos níveis iniciais.

O resultado do placebo, portanto, não aponta para uma violação evidente decorrente de tendências temporais distintas entre C1 e C2. Ainda assim, o teste placebo deve ser interpretado como uma evidência complementar, e não como uma prova definitiva da hipótese de tendências paralelas.

## Item 4 — DiD com painel completo

Nesta etapa, restringimos a análise aos domicílios observados tanto em 2005 quanto em 2009.

O objetivo é verificar se a estimativa de Diferenças em Diferenças é sensível à composição da amostra ao longo do tempo.

Na análise anterior, as médias de 2005 e 2009 podem ser calculadas a partir de conjuntos parcialmente diferentes de domicílios. Ao utilizar apenas unidades presentes nas duas ondas, passamos a comparar a evolução temporal de um conjunto fixo de domicílios.

Em seguida, reestimamos o DiD entre T1 e C1 para os três desfechos e comparamos os resultados com as estimativas obtidas na amostra original.

In [19]:
# ============================================================
# ITEM 4 — DOMICÍLIOS COM PAINEL COMPLETO
# ============================================================
#
# Queremos identificar os domicílios que aparecem
# nos dois períodos da análise:
#
# 2005 e 2009
#
# Para cada cod_dtm, contamos quantos anos distintos
# aparecem na base. Se o resultado for igual a 2,
# o domicílio está presente nas duas ondas.

anos_por_domicilio = (
    df
    .groupby("cod_dtm")["ano"]
    .nunique()
)

# Seleciona apenas os domicílios presentes em 2 anos
ids_painel_completo = anos_por_domicilio[
    anos_por_domicilio == 2
].index

print("Número de domicílios com painel completo:",
      len(ids_painel_completo))

Número de domicílios com painel completo: 6831


In [26]:
# Verifica se algum domicílio muda de grupo entre 2005 e 2009
mudanca_grupo = (
    df_painel
    .groupby("cod_dtm")["grupo"]
    .nunique()
    .value_counts()
)

display(mudanca_grupo)

,count
grupo,
1,6831


In [20]:
# Mantém somente observações pertencentes
# aos domicílios presentes em 2005 e 2009.

df_painel = df[
    df["cod_dtm"].isin(ids_painel_completo)
].copy()

print("Dimensões da base original:", df.shape)
print("Dimensões do painel completo:", df_painel.shape)

Dimensões da base original: (19425, 19)
Dimensões do painel completo: (13662, 19)


In [21]:
# Verifica quantas observações existem
# por grupo e por ano dentro do painel completo.

print("Grupo x ano — painel completo:")

display(
    pd.crosstab(
        df_painel["grupo"],
        df_painel["ano"]
    )
)

Grupo x ano — painel completo:


ano,2005,2009
grupo,,
C1,1427,1427
C2,3225,3225
T1,2179,2179


In [22]:
# Para reproduzir o DiD principal,
# restringimos o painel completo aos grupos T1 e C1.

did_painel = df_painel[
    df_painel["grupo"].isin(["T1", "C1"])
].copy()

print("Número de observações T1/C1 no painel completo:",
      did_painel.shape[0])

display(
    pd.crosstab(
        did_painel["grupo"],
        did_painel["ano"]
    )
)

Número de observações T1/C1 no painel completo: 7212


ano,2005,2009
grupo,,
C1,1427,1427
T1,2179,2179


In [23]:
# Calcula o DiD manualmente para os três desfechos,
# agora utilizando apenas os domicílios com painel completo.

for y in desfechos:

    tabela, did = tabela_did(
        did_painel,
        y
    )

    print(f"\n===== Painel completo: {y} =====")
    display(tabela)
    print(f"DiD = {did:.4f}")


===== Painel completo: attend_med =====


ano,2005,2009,Mudança
grupo,,,
C1,0.9326,0.8819,-0.0507
T1,0.9446,0.9002,-0.0445


DiD = 0.0062

===== Painel completo: dropout_med =====


ano,2005,2009,Mudança
grupo,,,
C1,0.0342,0.0896,0.0554
T1,0.0249,0.0743,0.0494


DiD = -0.0060

===== Painel completo: grade_now_med =====


ano,2005,2009,Mudança
grupo,,,
C1,3.6224,5.0434,1.4210
T1,3.6483,4.8952,1.2469


DiD = -0.1741


In [24]:
# Reestima a regressão DiD para os três desfechos
# utilizando somente os domicílios presentes
# em 2005 e 2009.

for y in desfechos:

    modelo = regressao_did(
        did_painel,
        y
    )

    print(f"\n===== Painel completo: {y} =====")
    print(
        "Coeficiente DiD:",
        round(modelo.params["tratado:pos"], 4)
    )
    print(
        "Erro-padrão:",
        round(modelo.bse["tratado:pos"], 4)
    )
    print(
        "p-valor:",
        round(modelo.pvalues["tratado:pos"], 4)
    )


===== Painel completo: attend_med =====
Coeficiente DiD: 0.0062
Erro-padrão: 0.0113
p-valor: 0.5834

===== Painel completo: dropout_med =====
Coeficiente DiD: -0.006
Erro-padrão: 0.01
p-valor: 0.5449

===== Painel completo: grade_now_med =====
Coeficiente DiD: -0.1741
Erro-padrão: 0.1028
p-valor: 0.0905


In [25]:
# ============================================================
# COMPARAÇÃO ENTRE A AMOSTRA ORIGINAL E O PAINEL COMPLETO
# ============================================================

# Valores obtidos anteriormente na amostra original
did_original = {
    "attend_med": 0.0089,
    "dropout_med": -0.0080,
    "grade_now_med": 0.1527
}

# Valores obtidos com o painel completo
did_painel_completo = {
    "attend_med": 0.0062,
    "dropout_med": -0.0060,
    "grade_now_med": -0.1741
}

# Organiza os resultados em uma tabela
comparacao = pd.DataFrame({
    "DiD_original": did_original,
    "DiD_painel_completo": did_painel_completo
})

# Calcula quanto a estimativa mudou
comparacao["Diferença"] = (
    comparacao["DiD_painel_completo"]
    - comparacao["DiD_original"]
)

display(comparacao)

,DiD_original,DiD_painel_completo,Diferença
attend_med,0.0089,0.0062,-0.0027
dropout_med,-0.0080,-0.0060,0.0020
grade_now_med,0.1527,-0.1741,-0.3268


### Interpretação do painel completo

A análise foi repetida restringindo a amostra aos 6.831 domicílios observados tanto em 2005 quanto em 2009, totalizando 13.662 observações.

Para `attend_med`, o DiD passou de 0,0089 para 0,0062. Para `dropout_med`, passou de -0,0080 para -0,0060. As alterações são pequenas, indicando que as estimativas pontuais desses dois desfechos são pouco sensíveis à restrição ao painel completo.

O comportamento de `grade_now_med` foi distinto. O DiD passou de 0,1527 na amostra original para -0,1741 no painel completo, com inversão de sinal. A diferença entre as duas estimativas foi de -0,3268.

Essa alteração sugere que a estimativa de `grade_now_med` é sensível à composição da amostra. Na especificação original, as médias de 2005 e 2009 podem envolver conjuntos parcialmente diferentes de domicílios. A restrição ao painel completo reduz essa fonte de mudança de composição ao acompanhar apenas unidades observadas nas duas ondas.

Na regressão com painel completo, os coeficientes de `attend_med` e `dropout_med` permaneceram estatisticamente indistinguíveis de zero. Para `grade_now_med`, o coeficiente foi -0,1741, com p-valor de 0,0905. Portanto, também não há evidência estatística suficiente para rejeitar a hipótese nula ao nível de 5%.

Os resultados indicam maior estabilidade para `attend_med` e `dropout_med` e maior sensibilidade amostral para `grade_now_med`. A análise com painel completo deve ser interpretada como uma verificação de robustez, pois a permanência dos domicílios nas duas ondas também pode estar associada a características específicas das unidades observadas.

## Considerações Finais

A análise de Diferenças em Diferenças foi realizada em quatro etapas.

Primeiro, o DiD canônico foi calculado manualmente para a comparação entre T1 e C1. As estimativas foram 0,0089 para `attend_med`, -0,0080 para `dropout_med` e 0,1527 para `grade_now_med`.

Em seguida, os mesmos efeitos foram estimados por regressão. Em todos os casos, o coeficiente da interação entre tratamento e período pós reproduziu exatamente os valores obtidos pela tabela 2×2. Os três coeficientes, contudo, apresentaram p-valores superiores a 0,05, de modo que não houve evidência estatística suficiente para rejeitar a hipótese nula de efeito igual a zero nos níveis convencionais.

No teste placebo entre C1 e C2, os efeitos estimados foram pequenos: -0,0010 para `attend_med`, 0,0086 para `dropout_med` e 0,0219 para `grade_now_med`. Nenhum deles apresentou significância estatística. Assim, o placebo não indicou uma diferença temporal especialmente suspeita entre os dois grupos não tratados.

Por fim, a análise foi repetida utilizando apenas os 6.831 domicílios observados em 2005 e 2009. Os resultados de `attend_med` e `dropout_med` permaneceram próximos dos obtidos na amostra original. Já `grade_now_med` apresentou forte sensibilidade à composição da amostra: o DiD passou de 0,1527 para -0,1741, com inversão de sinal.

Esse resultado sugere que a estimativa de `grade_now_med` depende de forma relevante da composição dos domicílios observados nas duas ondas, enquanto os resultados de `attend_med` e `dropout_med` se mostraram mais estáveis. A restrição ao painel completo reduz o problema de mudança de composição entre os períodos, mas também seleciona apenas os domicílios observados nas duas ondas, razão pela qual deve ser interpretada como uma análise de robustez, e não como uma especificação necessariamente superior.